# Financial MCQ Prompt Engineering Study: Fin-R1 (7B)

Single-strategy-per-run design. Set `ACTIVE_STRATEGY` in Cell 1, run on Kaggle,
download the JSON, and compare all strategies offline.

**Model**: `SUFE-AIFLM-Lab/Fin-R1` (7B, finance fine-tuned, 4-bit NF4 quantisation)  
**Strategies**: baseline | few_shot | cot | role | role_cot | self_consistency | meta  
**Datasets**: cfa_cpa | es_multifin | plutus | arabic_accounting | arabic_business | hindi_finance  
**Sampling**: Balanced — equal rows from each dataset (min dataset size)  
**Metric**: Accuracy — predicted letter in gold list (multi-answer aware)

## Cell 1 — Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ACTIVE STRATEGY SELECTOR
# Set this to run only ONE strategy per Kaggle session.
# Run multiple sessions in parallel, each with a different strategy,
# then download all JSON files and compare offline.
#
# Valid values:
#   "baseline"         — zero-shot, letter only (logit scoring)
#   "few_shot"         — 3 in-context examples (logit scoring)
#   "cot"              — chain-of-thought (generation)
#   "strict_cot"       — CoT forcing Fin-R1 native <think>/<answer> format (recommended over cot)
#   "role"             — expert CFA persona (logit scoring)
#   "role_cot"         — expert persona + CoT (generation)
#   "self_consistency" — majority vote over SC_SAMPLES samples (generation)
#   "meta"             — elimination-based reasoning (generation)
# ═══════════════════════════════════════════════════════════════════════════
ACTIVE_STRATEGY = "baseline"   # ← CHANGE THIS per Kaggle session

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit before running
# ═══════════════════════════════════════════════════════════════════════════

import random

# Reproducibility — fixes few-shot sampling and self-consistency across runs
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# TEST MODE — set True to run exactly 1 prediction per dataset (6 total)
# to verify the full pipeline end-to-end in seconds. Set False for a full run.
TEST_MODE = True     # ← True = pipeline smoke-test (1 sample/dataset) | False = full run
N_TEST    = 1        # 1 prediction per dataset when TEST_MODE is True

# Prompt strategies — runs only the active strategy selected above
STRATEGIES = [ACTIVE_STRATEGY]

# Self-consistency: number of independent samples per question
SC_SAMPLES = 5

# Fin-R1 is 7B (4-bit ≈ 4 GB VRAM) vs Qwen 14B (4-bit ≈ 8 GB).
# Larger batch sizes are safe on T4 (16 GB) and P100 (16 GB).
# If you hit OOM, halve BATCH_SIZE and LOGIT_BATCH_SIZE.
BATCH_SIZE       = 8    # generation strategies (cot, role_cot, meta)
LOGIT_BATCH_SIZE = 2    # logit strategies (baseline, few_shot, role)
SC_BATCH_SIZE    = 1    # SC expands to SC_SAMPLES sequences — keep 1 on T4

# Max new tokens per strategy.
# strict_cot needs a large budget so the model can complete its full
# <think>...</think><answer>X</answer> chain before being cut off.
# 256 is enough for cot/role_cot/meta; 1024 is the safe minimum for strict_cot.
# Lower this if you hit OOM (e.g. 512), but don't go below ~400.
MAX_NEW_TOKENS_DEFAULT  = 256    # cot, role_cot, meta
MAX_NEW_TOKENS_STRICT_COT = 1024  # strict_cot — must finish <think> block

# Max tokeniser length — raised to 4096 to avoid truncating long CoT / few-shot prompts.
MAX_LENGTH = 4096

# Model
MODEL_ID = "SUFE-AIFLM-Lab/Fin-R1"

print(f"ACTIVE_STRATEGY           : {ACTIVE_STRATEGY}")
print(f"RANDOM_SEED               : {RANDOM_SEED}")
print(f"TEST_MODE                 : {TEST_MODE}")
if TEST_MODE:
    print(f"N_TEST                    : {N_TEST} examples per dataset")
print(f"STRATEGIES                : {STRATEGIES}")
print(f"BATCH_SIZE                : {BATCH_SIZE}  (cot / role_cot / meta)")
print(f"LOGIT_BATCH_SIZE          : {LOGIT_BATCH_SIZE}  (baseline / few_shot / role)")
print(f"SC_BATCH_SIZE             : {SC_BATCH_SIZE}  (self_consistency — eff. batch = {SC_BATCH_SIZE}×{SC_SAMPLES}={SC_BATCH_SIZE*SC_SAMPLES})")
print(f"SC_SAMPLES                : {SC_SAMPLES}")
print(f"MAX_NEW_TOKENS_DEFAULT    : {MAX_NEW_TOKENS_DEFAULT}  (cot / role_cot / meta)")
print(f"MAX_NEW_TOKENS_STRICT_COT : {MAX_NEW_TOKENS_STRICT_COT}  (strict_cot)")
print(f"MAX_LENGTH                : {MAX_LENGTH}")
print(f"MODEL                     : {MODEL_ID}")


## Cell 2 — Install Required Libraries

In [ ]:
!pip install transformers datasets peft bitsandbytes accelerate
!pip install sentencepiece protobuf evaluate scikit-learn tqdm
!pip install sacremoses langdetect
!pip install -U bitsandbytes>=0.46.1 accelerate transformers

## Cell 3 — HuggingFace Login

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
login(token=hf_token)
print("HuggingFace login OK")

## Cell 4 — Load & Balance Datasets from GitHub

Pre-cleaned task1 JSON files fetched from `hassan09070/clef_task/task1_data/`.  
Large files (>1 MB) are fetched via `download_url` streaming fallback.

Row schema:
- `question`: str
- `choices`: list of N strings
- `keys`: list of lowercase letter keys matching choices
- `gold`: list of correct letter(s)
- `source`: dataset identifier

**Balanced sampling**: after loading, the smallest dataset size is computed and
every dataset is randomly sampled down to that size so all sources contribute equally.

In [ ]:
import requests
import json
import base64
import os

# ── GitHub source config ───────────────────────────────────────────────────────────
_GITHUB_REPO   = "hassan09070/clef_task"
_GITHUB_FOLDER = "task1_data"
_GITHUB_TOKEN  = "YOUR_GITHUB_TOKEN_HERE"


def _get_github_token() -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("github_token")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("GITHUB_TOKEN") or _GITHUB_TOKEN


def _github_fetch(filename: str) -> list:
    """
    Fetch a JSON file from the private GitHub repo.
    Handles large files (>1 MB) via download_url fallback.
    """
    token   = _get_github_token()
    headers = {
        "Accept":        "application/vnd.github+json",
        "Authorization": f"token {token}",
    }
    api_url = (
        f"https://api.github.com/repos/{_GITHUB_REPO}/contents/"
        f"{_GITHUB_FOLDER}/{filename}"
    )
    meta = requests.get(api_url, headers=headers, timeout=60)
    meta.raise_for_status()
    data = meta.json()

    content_b64 = data.get("content", "").replace("\n", "")
    if content_b64:
        return json.loads(base64.b64decode(content_b64).decode("utf-8"))

    # Large file path (>1 MB)
    download_url = data.get("download_url")
    if not download_url:
        raise ValueError(f"No content or download_url for {filename}")
    print(f"    [large file] streaming via download_url ...")
    chunks = []
    raw = requests.get(
        download_url,
        headers={"Authorization": f"token {token}"},
        timeout=300,
        stream=True,
    )
    raw.raise_for_status()
    total = 0
    for chunk in raw.iter_content(chunk_size=65536):
        chunks.append(chunk)
        total += len(chunk)
    print(f"    [large file] downloaded {total // 1024} KB")
    return json.loads(b"".join(chunks).decode("utf-8"))


# ── Dataset file mapping ───────────────────────────────────────────────────────────────
_DATASET_FILES = {
    "cfa_cpa":           "task1_Tomas08119993_finmmeval-cfa-cpa.json",
    "es_multifin":       "task1_TheFinAI_flare-es-multifin.json",
    "plutus":            "task1_TheFinAI_plutus-multifin.json",
    "arabic_accounting": "task1_SahmBenchmark_arabic-accounting-mcq.json",
    "arabic_business":   "task1_SahmBenchmark_arabic-business-mcq.json",
    "hindi_finance":     "task1_bharatgenai_BhashaBench-Finance-Hindi.json",
}


def load_task1_json(filename: str, source_name: str) -> list:
    records = _github_fetch(filename)
    rows    = []
    for rec in records:
        options = rec.get("options") or {}
        if not options:
            continue
        sorted_keys = sorted(options.keys())
        choices     = [options[k] for k in sorted_keys]
        gold_raw    = rec.get("gold") or []
        valid_gold  = [g.lower() for g in gold_raw if g.lower() in sorted_keys]
        if not valid_gold:
            continue
        rows.append({
            "question": str(rec.get("question") or ""),
            "choices":  choices,
            "keys":     sorted_keys,
            "gold":     valid_gold,
            "source":   source_name,
        })
    return rows


# ── Load all datasets ────────────────────────────────────────────────────────────────────
print(f"Loading datasets from github.com/{_GITHUB_REPO} ...")
print()

datasets_raw = {}   # {source_name: list_of_rows} — kept for few-shot example pool

for src_name, fname in _DATASET_FILES.items():
    print(f"  Loading {src_name} ...", flush=True)
    try:
        rows = load_task1_json(fname, src_name)
        datasets_raw[src_name] = rows
        print(f"  {src_name:<33} {len(rows):>5} rows  OK")
    except Exception as e:
        print(f"  {src_name:<33} FAILED — {e}")

print()

# ── After loading all datasets_raw ────────────────────────────────────────────────────
# BALANCED SAMPLING:
# Find the dataset with the fewest rows. Sample that many rows from every
# other dataset so all datasets contribute equally.
# In TEST_MODE, cap each dataset at min(N_TEST, min_dataset_size).

if TEST_MODE:
    # First trim each to N_TEST so TEST_MODE is still fast
    for src in datasets_raw:
        datasets_raw[src] = datasets_raw[src][:N_TEST]

# Find the smallest dataset size
min_size = min(len(rows) for rows in datasets_raw.values())
print(f"Smallest dataset has {min_size} rows — sampling {min_size} from each dataset")

# Sample min_size rows from each dataset with fixed seed for reproducibility
_balance_rng = random.Random(RANDOM_SEED)
all_data = []
for src_name, rows in datasets_raw.items():
    sampled = _balance_rng.sample(rows, min_size) if len(rows) > min_size else rows
    datasets_raw[src_name] = sampled   # update in place so few-shot pool is also balanced
    all_data.extend(sampled)
    print(f"  {src_name:<33}  {len(sampled):>5} rows  (balanced)")

print()
print(f"Total after balancing: {len(all_data)} examples  ({min_size} × {len(datasets_raw)} datasets)")

## Cell 5 — Dataset Stats

In [ ]:
from collections import Counter

# Language tag per source — used for narrative / analysis
_SOURCE_LANG = {
    "cfa_cpa":           "English",
    "es_multifin":       "Spanish",
    "plutus":            "Multilingual",
    "arabic_accounting": "Arabic",
    "arabic_business":   "Arabic",
    "hindi_finance":     "Hindi",
}

print(f"Total examples: {len(all_data)}")
print(f"TEST_MODE active: {TEST_MODE}" + (f"  ({N_TEST} per dataset)" if TEST_MODE else ""))
print()

print(f"{'Source':<33}  {'Lang':<13}  {'Rows':>5}  Choice distribution")
print("-" * 78)
for src in sorted(datasets_raw.keys()):
    rows      = datasets_raw[src]
    n_choices = Counter(len(r["choices"]) for r in rows)
    choices_s = "  ".join(f"{k}opt:{v}" for k, v in sorted(n_choices.items()))
    lang      = _SOURCE_LANG.get(src, "?")
    print(f"  {src:<31}  {lang:<13}  {len(rows):>5}  ({choices_s})")

multi_gold = sum(1 for r in all_data if len(r["gold"]) > 1)
lang_counts = Counter(_SOURCE_LANG.get(r["source"], "?") for r in all_data)
print()
print("Language distribution:")
for lang, cnt in sorted(lang_counts.items(), key=lambda x: -x[1]):
    pct = cnt / len(all_data) * 100
    print(f"  {lang:<15}  {cnt:>5}  ({pct:.1f}%)")
print(f"\nMulti-gold rows (>1 correct answer): {multi_gold}")

## Cell 6 — Prompt Engineering Strategies

Implementations of all 7 prompting styles.  

| Strategy | Inference method | Key idea |
|---|---|---|
| `baseline` | logit scoring | Letter-only zero-shot |
| `few_shot` | logit scoring | 3 in-context examples (seeded for reproducibility) |
| `cot` | generation | "Think step by step" then extract final letter |
| `role` | logit scoring | Expert CFA persona — logit-only for speed |
| `role_cot` | generation | Same CFA persona but with CoT reasoning step |
| `self_consistency` | generation + vote | SC_SAMPLES samples → majority vote |
| `meta` | generation | Explicit elimination-based reasoning strategy |

**Design notes:**
- `role` uses logit scoring because the persona is injected via the system prompt — the model's option probabilities already reflect the expert framing without needing free-text generation.
- `role_cot` is the generation companion: it lets the model verbalise its reasoning under the expert persona. Use it to test whether CoT helps on top of role prompting.
- Few-shot examples are sampled with a fixed `random.Random(RANDOM_SEED)` instance so results are fully reproducible across Kaggle re-runs, regardless of the global `random` state.

In [ ]:
import random
import re

# Dedicated RNG for few-shot sampling — isolated from the global random state
# so that few-shot examples are always the same regardless of call order.
_FEWSHOT_RNG = random.Random(RANDOM_SEED)

# ── Shared helpers ───────────────────────────────────────────────────────────────────────

def _labels_and_options(choices: list) -> tuple:
    """Return (display_labels, options_str, valid_letters_str)."""
    n      = len(choices)
    labels = [chr(ord("A") + i) for i in range(n)]
    opts   = "\n".join(f"{labels[i]}. {choices[i]}" for i in range(n))
    valid  = "/".join(labels)
    return labels, opts, valid


def _apply_chat_template(tokenizer, system_msg: str, user_msg: str) -> str:
    """Apply chat template if available, fall back to INST format."""
    if getattr(tokenizer, "chat_template", None):
        messages = []
        if system_msg:
            messages.append({"role": "system", "content": system_msg})
        messages.append({"role": "user", "content": user_msg})
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    prefix = f"<<SYS>>{system_msg}<</SYS>>\n\n" if system_msg else ""
    return f"[INST] {prefix}{user_msg} [/INST]"


def _extract_letter(text: str, valid_labels: list) -> str | None:
    """
    Extract the predicted letter from generated text.

    Priority order (highest to lowest confidence):
    1.  <answer>X</answer>         — Fin-R1 native format (MUST be checked first)
    1b. <answer>\\boxed{X}</answer> — Fin-R1 LaTeX boxing variant
    2.  <answer>X                  — unclosed tag (model truncated)
    3.  "Answer: X"                — explicit marker
    4.  "The answer is X"          — natural language marker
    5.  "Final answer: X"          — final answer marker
    6.  "(X)" or "[X]"             — bracketed letter
    7.  "**X**"                    — bold letter
    8.  Trailing letter            — last letter on final line
    9.  First valid letter         — last resort fallback (least reliable)

    All checks are case-insensitive. Returns lowercase key or None.
    valid_labels must be UPPERCASE (e.g. ["A","B","C","D"]).
    """
    text_up = text.upper().strip()

    # 1. <answer>X</answer> — Fin-R1 native format — highest priority
    m = re.search(r"<ANSWER>\s*([A-Z])\s*</ANSWER>", text_up)
    if m and m.group(1) in valid_labels:
        return m.group(1).lower()

    # 1b. <answer>\boxed{X}</answer> — Fin-R1 sometimes uses LaTeX boxing
    m = re.search(r"<ANSWER>\s*\\BOXED\{([A-Z])\}\s*</ANSWER>", text_up)
    if m and m.group(1) in valid_labels:
        return m.group(1).lower()

    # 2. <answer>X (unclosed — model was cut off before </answer>)
    m = re.search(r"<ANSWER>\s*([A-Z])", text_up)
    if m and m.group(1) in valid_labels:
        return m.group(1).lower()

    # 3-5. Explicit natural language markers
    for pat in [
        r"ANSWER\s*:\s*([A-Z])",
        r"THE\s+ANSWER\s+IS\s*:?\s*([A-Z])",
        r"FINAL\s+ANSWER\s*:\s*([A-Z])",
        r"CORRECT\s+ANSWER\s*:\s*([A-Z])",
        r"THEREFORE[,\s]+THE\s+ANSWER\s+IS\s*:?\s*([A-Z])",
    ]:
        m = re.search(pat, text_up)
        if m and m.group(1) in valid_labels:
            return m.group(1).lower()

    # 6. Bracketed letter: (B) or [B]
    for pat in [r"\(([A-Z])\)", r"\[([A-Z])\]"]:
        m = re.search(pat, text_up)
        if m and m.group(1) in valid_labels:
            return m.group(1).lower()

    # 7. Bold markdown **X**
    m = re.search(r"\*\*([A-Z])\*\*", text_up)
    if m and m.group(1) in valid_labels:
        return m.group(1).lower()

    # 8. Trailing letter on the last non-empty line
    for line in reversed(text_up.splitlines()):
        line = line.strip()
        if not line:
            continue
        m = re.search(r"[.\s]\s*([A-Z])\s*$", line)
        if m and m.group(1) in valid_labels:
            return m.group(1).lower()
        break

    # 9. Last resort: first valid letter found anywhere (very unreliable)
    for ch in text_up:
        if ch in valid_labels:
            return ch.lower()

    return None


# ── Few-shot example pool ────────────────────────────────────────────────────────

def _build_few_shot_examples(source: str, exclude_idx: int, n: int = 3) -> str:
    """
    Pick n examples from the same source (excluding `exclude_idx`) using
    _FEWSHOT_RNG so results are reproducible regardless of global random state.

    Finance MCQ datasets sometimes contain near-duplicate questions.
    We guard against direct index repetition (exclude_idx) but do not do
    semantic deduplication — this is an acknowledged limitation. For full
    deduplication, a sentence-embedding similarity check would be needed.

    Falls back to the full dataset pool if the same-source pool is too small.
    """
    pool = [
        r for i, r in enumerate(all_data)
        if r["source"] == source and i != exclude_idx
    ]
    if len(pool) < n:
        pool = [r for i, r in enumerate(all_data) if i != exclude_idx]
    sampled = _FEWSHOT_RNG.sample(pool, min(n, len(pool)))
    parts   = []
    for ex in sampled:
        labels, opts, _ = _labels_and_options(ex["choices"])
        answer_letter   = ex["gold"][0].upper()
        parts.append(f"Question: {ex['question']}\n{opts}\nAnswer: {answer_letter}")
    return "\n\n".join(parts)


# ── CFA expert system prompt (shared by role and role_cot) ────────────────────────
_ROLE_SYSTEM = (
    "You are a senior Chartered Financial Analyst (CFA) with over 20 years of "
    "experience in financial markets, accounting standards, and investment analysis. "
    "You have deep expertise in international finance, business law, and economics. "
    "When answering questions, draw on your professional expertise and provide the "
    "most accurate financial judgement."
)

# ═══════════════════════════════════════════════════════════════════════════
# Prompt builders — one per strategy
# Each accepts **kwargs so new callers can pass (source=, row_idx=) freely.
# ═══════════════════════════════════════════════════════════════════════════

def build_baseline_prompt(tokenizer, question, choices, **_) -> str:
    labels, opts, valid = _labels_and_options(choices)
    user = (
        "Answer the following multiple-choice question. "
        f"Respond with ONLY the single letter of the correct answer ({valid}).\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _apply_chat_template(tokenizer, "", user)


def build_few_shot_prompt(tokenizer, question, choices, source, row_idx, **_) -> str:
    labels, opts, valid = _labels_and_options(choices)
    examples = _build_few_shot_examples(source, row_idx, n=3)
    user = (
        "Answer financial multiple-choice questions. "
        f"Respond with ONLY the single letter ({valid}).\n\n"
        "Here are three solved examples:\n\n"
        f"{examples}\n\n"
        "Now answer this question:\n\n"
        f"Question: {question}\n\n{opts}\nAnswer:"
    )
    return _apply_chat_template(tokenizer, "", user)


def build_cot_prompt(tokenizer, question, choices, **_) -> str:
    labels, opts, valid = _labels_and_options(choices)
    user = (
        "Answer the following financial multiple-choice question. "
        "Think through the problem step by step, then state your final answer "
        f"on a new line as: Answer: <letter>  (where letter is one of {valid}).\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _apply_chat_template(tokenizer, "", user)


def build_strict_cot_prompt(tokenizer, question, choices, **_) -> str:
    """
    Strict CoT prompt that explicitly demands Fin-R1's native output format:
        <think>reasoning</think><answer>LETTER</answer>

    This matches what Fin-R1 was trained to produce, making extraction
    reliable. Use this instead of build_cot_prompt when running CoT on Fin-R1.
    """
    labels, opts, valid = _labels_and_options(choices)
    user = (
        f"Answer this financial multiple-choice question. "
        f"Valid answer letters: {valid}.\n\n"
        f"Question: {question}\n\n{opts}\n\n"
        f"You MUST use this exact format:\n"
        f"<think>\n"
        f"[your step-by-step reasoning here]\n"
        f"</think>\n"
        f"<answer>\n"
        f"[single letter only — one of {valid}]\n"
        f"</answer>"
    )
    return _apply_chat_template(tokenizer, "", user)


def build_role_prompt(tokenizer, question, choices, **_) -> str:
    """
    Role prompting with logit scoring.
    The expert persona is injected as a system message; the model's option
    probabilities at the last input token already reflect the expert framing.
    No free-text generation needed — fast and deterministic.
    """
    labels, opts, valid = _labels_and_options(choices)
    user = (
        f"Answer this multiple-choice question. Respond with ONLY the letter ({valid}).\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _apply_chat_template(tokenizer, _ROLE_SYSTEM, user)


def build_role_cot_prompt(tokenizer, question, choices, **_) -> str:
    """
    Role prompting + Chain-of-Thought (generation variant).
    Same expert persona as role, but asks the model to reason step by step
    and emit a final 'Answer: X' line — extracted by _extract_letter.
    Tests whether explicit reasoning adds value on top of the persona alone.
    """
    labels, opts, valid = _labels_and_options(choices)
    user = (
        "Using your expert financial knowledge, think through this question "
        "step by step. Consider each option carefully, then state your final "
        f"answer on a new line as: Answer: <letter>  (one of {valid}).\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _apply_chat_template(tokenizer, _ROLE_SYSTEM, user)


def build_meta_prompt(tokenizer, question, choices, **_) -> str:
    labels, opts, valid = _labels_and_options(choices)
    user = (
        "You will answer a financial multiple-choice question. "
        "Use the following reasoning strategy:\n"
        "1. Read all options carefully.\n"
        "2. Eliminate options that are clearly incorrect.\n"
        "3. For the remaining options, identify which one is most precisely correct "
        "according to standard financial principles.\n"
        "4. State your final answer as: Answer: <letter>\n\n"
        f"Valid answer letters: {valid}\n\n"
        f"Question: {question}\n\n{opts}"
    )
    return _apply_chat_template(tokenizer, "", user)


def build_self_consistency_prompt(tokenizer, question, choices, **_) -> str:
    """Self-consistency reuses the baseline prompt; diversity comes from sampling."""
    return build_baseline_prompt(tokenizer, question, choices)


# Registry
PROMPT_BUILDERS = {
    "baseline":         build_baseline_prompt,
    "few_shot":         build_few_shot_prompt,
    "cot":              build_cot_prompt,          # original (kept for comparison)
    "strict_cot":       build_strict_cot_prompt,   # NEW — forces <answer> format
    "role":             build_role_prompt,
    "role_cot":         build_role_cot_prompt,
    "self_consistency": build_self_consistency_prompt,
    "meta":             build_meta_prompt,
}

print("Prompt builders registered:")
for name, fn in PROMPT_BUILDERS.items():
    print(f"  {name}")


## Cell 7 — Model Loading & GPU Utilities

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc


def load_model_4bit(model_name: str):
    """Load causal LM in 4-bit NF4 quantisation."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True,  # allows fp32 modules (e.g. lm_head) to offload to CPU
    )
    print(f"[model] loading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"[model] loading weights (4-bit NF4) ...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print(f"[model] loaded OK: {model_name}")
    return model, tokenizer


def free_model(model):
    """Release model from GPU memory."""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("[model] released from GPU memory")


print("Model utilities ready (load_model_4bit, free_model)")

## Cell 8 — Batch Inference Engine

Handles all 7 strategies with batched GPU inference.

**Logit strategies** (`baseline`, `few_shot`, `role`):  
Scores each option label at the last input token position — no generation required.  
Batched: `LOGIT_BATCH_SIZE` prompts per forward pass, left-padded.

**Generation strategies** (`cot`, `role_cot`, `meta`):  
Generates up to 256 tokens of free text, then extracts the final letter via regex.  
Batched: `BATCH_SIZE` prompts per forward pass, left-padded.

**Self-consistency** (`self_consistency`):  
Batched SC: processes `SC_BATCH_SIZE` questions simultaneously, each expanded to `SC_SAMPLES` rows,  
giving `SC_BATCH_SIZE × SC_SAMPLES` total rows per forward pass. Majority-votes across samples per question.  
Falls back to logit scoring if no valid letter is extracted from any sample.

**Error logging:** `run_strategy` also returns an `errors` list —  
`[{idx, source, gold, predicted, strategy}, ...]` — for every mispredicted question.  
Used downstream in the error analysis / save cells.

In [ ]:
import gc
from collections import Counter as _Counter

_LOGIT_STRATEGIES    = {"baseline", "few_shot", "role"}
_GENERATE_STRATEGIES = {"cot", "role_cot", "meta", "strict_cot"}
_SC_STRATEGY         = "self_consistency"


def _get_label_token_ids(tokenizer, labels: list) -> dict:
    return {lbl: tokenizer.convert_tokens_to_ids(lbl) for lbl in labels}


def _flush_cuda():
    """Release Python-unreachable tensors and clear the CUDA memory pool."""
    gc.collect()
    torch.cuda.empty_cache()


def _logit_fallback(model, tokenizer, prompt: str, labels: list, keys: list) -> str:
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    enc = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH,
    ).to(model.device)
    with torch.no_grad():
        try:
            logits = model(**enc, use_cache=False, logits_to_keep=1).logits
        except TypeError:
            logits = model(**enc, use_cache=False).logits
    tok_ids = _get_label_token_ids(tokenizer, labels)
    best_l  = max(labels, key=lambda l: logits[0, -1, tok_ids[l]].item())
    result  = keys[labels.index(best_l)]
    del enc, logits
    _flush_cuda()
    return result


# ── Logit-based ───────────────────────────────────────────────────────────────────────

def _batch_logit_predict(model, tokenizer, prompts: list, labels_list: list, keys_list: list) -> list:
    """
    LEFT-PADDING + logits_to_keep=1.
    Frees enc + activations immediately after the forward pass, then flushes
    the CUDA allocator so nothing leaks between batches.
    """
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    enc = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        try:
            out = model(**enc, use_cache=False, logits_to_keep=1)
        except TypeError:
            out = model(**enc, use_cache=False)
    logits = out.logits
    del out, enc
    _flush_cuda()    # ← flush #1: free all forward-pass activations

    preds = []
    for b_idx in range(len(prompts)):
        last_logits = logits[b_idx, -1, :]
        labels      = labels_list[b_idx]
        keys        = keys_list[b_idx]
        tok_ids     = _get_label_token_ids(tokenizer, labels)
        best_label  = max(labels, key=lambda l: last_logits[tok_ids[l]].item())
        preds.append(keys[labels.index(best_label)])
    del logits
    _flush_cuda()    # ← flush #2: free the logit tensor itself
    return preds


# ── Generation-based ────────────────────────────────────────────────────────────────────

def _batch_generate_predict(model, tokenizer, prompts: list, labels_list: list, keys_list: list,
                             max_new_tokens: int = 256) -> tuple:
    """
    Greedy generation. temperature/top_p/top_k=None suppresses generation_config warning.
    Returns (preds, raw_texts) — raw_texts holds the decoded model output per question,
    including any <think>...</think> / <answer> blocks, for offline inspection.
    """
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    enc = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            use_cache=True,
        )

    input_len = enc["input_ids"].shape[1]
    del enc
    _flush_cuda()    # ← flush #1: free input tensors

    preds     = []
    raw_texts = []
    for b_idx in range(len(prompts)):
        new_ids  = out_ids[b_idx, input_len:]
        text     = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
        labels   = labels_list[b_idx]
        keys     = keys_list[b_idx]
        pred_key = _extract_letter(text, labels)
        if pred_key is None:
            pred_key = _logit_fallback(model, tokenizer, prompts[b_idx], labels, keys)
        preds.append(pred_key)
        raw_texts.append(text)
    del out_ids
    _flush_cuda()    # ← flush #2: free generated token tensor
    return preds, raw_texts


# ── Self-consistency ────────────────────────────────────────────────────────────────────

def _sc_batch_predict(model, tokenizer, prompts: list, labels_list: list, keys_list: list,
                      n_samples: int) -> tuple:
    """
    SC_BATCH_SIZE questions × n_samples sequences per forward pass.
    On T4: SC_BATCH_SIZE=1 → 5 sequences per pass (safe).
    Returns (preds, raw_texts) — raw_texts[q] is a list of SC_SAMPLES decoded strings
    for question q, so the full reasoning diversity is visible in the saved JSON.
    """
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    repeated_prompts = [p for p in prompts for _ in range(n_samples)]

    enc = tokenizer(
        repeated_prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **enc,
            max_new_tokens=64,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            use_cache=True,
        )

    input_len = enc["input_ids"].shape[1]
    del enc
    _flush_cuda()    # ← flush #1: free input tensors

    preds     = []
    raw_texts = []
    for q_idx in range(len(prompts)):
        labels       = labels_list[q_idx]
        keys         = keys_list[q_idx]
        votes        = []
        sample_texts = []
        for s in range(n_samples):
            row_idx = q_idx * n_samples + s
            new_ids = out_ids[row_idx, input_len:]
            text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            sample_texts.append(text)
            letter  = _extract_letter(text, labels)
            if letter:
                votes.append(letter)
        raw_texts.append(sample_texts)
        if votes:
            preds.append(_Counter(votes).most_common(1)[0][0])
        else:
            preds.append(_logit_fallback(model, tokenizer, prompts[q_idx], labels, keys))
    del out_ids
    _flush_cuda()    # ← flush #2: free generated token tensor
    return preds, raw_texts


# ═══════════════════════════════════════════════════════════════════════════
# Main runner
# ═══════════════════════════════════════════════════════════════════════════

def run_strategy(model, tokenizer, data: list, strategy: str) -> tuple:
    """
    Returns (preds, errors, raw_outputs).
    raw_outputs[i] holds the decoded model text for question i:
      - generation strategies : str  (full model output including <think> etc.)
      - self_consistency       : list[str]  (one string per SC sample)
      - logit strategies       : None  (no free-text generation)

    Token budget per strategy:
      - strict_cot uses MAX_NEW_TOKENS_STRICT_COT (default 1024) so the model
        can complete its full <think>…</think><answer>X</answer> block.
      - all other generation strategies use MAX_NEW_TOKENS_DEFAULT (default 256).

    Memory management:
      - _flush_cuda() at the top clears leftovers from the previous strategy.
      - Each batch function also flushes internally (twice: after fwd pass, after del).
      - _flush_cuda() after every batch call in this loop ensures the CUDA allocator
        is clean before the next batch regardless of Python GC timing.
        This is critical at 8k data points where fragmentation accumulates.
    """
    _flush_cuda()    # ← clear leftovers from previous strategy

    builder = PROMPT_BUILDERS[strategy]
    n       = len(data)
    preds       = [None] * n
    raw_outputs = [None] * n   # ← stores decoded model text per question

    # Per-strategy token budget — strict_cot must finish its <think> block
    if strategy == "strict_cot":
        gen_max_new = MAX_NEW_TOKENS_STRICT_COT
    else:
        gen_max_new = MAX_NEW_TOKENS_DEFAULT

    if strategy == _SC_STRATEGY:
        bsz = SC_BATCH_SIZE
    elif strategy in _LOGIT_STRATEGIES:
        bsz = LOGIT_BATCH_SIZE
    else:
        bsz = BATCH_SIZE

    print(f"  [{strategy}] starting — {n} questions  (batch_size={bsz}, max_new_tokens={gen_max_new if strategy in _GENERATE_STRATEGIES else 'n/a'})", flush=True)

    if strategy == _SC_STRATEGY:
        for batch_start in range(0, n, SC_BATCH_SIZE):
            batch      = data[batch_start: batch_start + SC_BATCH_SIZE]
            b_prompts, b_labels, b_keys = [], [], []
            for idx, row in enumerate(batch):
                labels, _, _ = _labels_and_options(row["choices"])
                b_prompts.append(builder(
                    tokenizer, question=row["question"], choices=row["choices"],
                    source=row["source"], row_idx=batch_start + idx,
                ))
                b_labels.append(labels)
                b_keys.append(row["keys"])
            batch_preds, batch_raw = _sc_batch_predict(model, tokenizer, b_prompts, b_labels, b_keys, SC_SAMPLES)
            for k, p in enumerate(batch_preds):
                preds[batch_start + k]       = p
                raw_outputs[batch_start + k] = batch_raw[k]
            _flush_cuda()   # ← flush after every batch in the loop
            done    = batch_start + len(batch)
            correct = sum(1 for j in range(done) if preds[j] in data[j]["gold"])
            print(f"  [{strategy}] [{done:>5}/{n}]  running acc: {correct/done*100:.1f}%", flush=True)

    elif strategy in _GENERATE_STRATEGIES:
        for batch_start in range(0, n, BATCH_SIZE):
            batch      = data[batch_start: batch_start + BATCH_SIZE]
            b_prompts, b_labels, b_keys = [], [], []
            for idx, row in enumerate(batch):
                labels, _, _ = _labels_and_options(row["choices"])
                b_prompts.append(builder(
                    tokenizer, question=row["question"], choices=row["choices"],
                    source=row["source"], row_idx=batch_start + idx,
                ))
                b_labels.append(labels)
                b_keys.append(row["keys"])
            batch_preds, batch_raw = _batch_generate_predict(
                model, tokenizer, b_prompts, b_labels, b_keys,
                max_new_tokens=gen_max_new,
            )
            for k, p in enumerate(batch_preds):
                preds[batch_start + k]       = p
                raw_outputs[batch_start + k] = batch_raw[k]
            _flush_cuda()   # ← flush after every batch in the loop
            done    = batch_start + len(batch)
            correct = sum(1 for j in range(done) if preds[j] in data[j]["gold"])
            print(f"  [{strategy}] [{done:>5}/{n}]  running acc: {correct/done*100:.1f}%", flush=True)

    else:
        for batch_start in range(0, n, LOGIT_BATCH_SIZE):
            batch      = data[batch_start: batch_start + LOGIT_BATCH_SIZE]
            b_prompts, b_labels, b_keys = [], [], []
            for idx, row in enumerate(batch):
                labels, _, _ = _labels_and_options(row["choices"])
                b_prompts.append(builder(
                    tokenizer, question=row["question"], choices=row["choices"],
                    source=row["source"], row_idx=batch_start + idx,
                ))
                b_labels.append(labels)
                b_keys.append(row["keys"])
            batch_preds = _batch_logit_predict(model, tokenizer, b_prompts, b_labels, b_keys)
            for k, p in enumerate(batch_preds):
                preds[batch_start + k] = p
            _flush_cuda()   # ← flush after every batch in the loop
            done    = batch_start + len(batch)
            correct = sum(1 for j in range(done) if preds[j] in data[j]["gold"])
            print(f"  [{strategy}] [{done:>5}/{n}]  running acc: {correct/done*100:.1f}%", flush=True)

    correct = sum(1 for i, p in enumerate(preds) if p in data[i]["gold"])
    acc     = correct / n * 100
    print(f"  [{strategy}] FINAL — acc: {acc:.2f}%  ({correct}/{n})", flush=True)

    errors = [
        {
            "idx":       i,
            "source":    data[i]["source"],
            "gold":      data[i]["gold"],
            "predicted": preds[i],
            "strategy":  strategy,
        }
        for i, p in enumerate(preds)
        if p not in data[i]["gold"]
    ]
    return preds, errors, raw_outputs


print("Batch inference engine ready.")
print(f"  baseline/few_shot/role : LOGIT_BATCH_SIZE={LOGIT_BATCH_SIZE}")
print(f"  cot/role_cot/meta      : BATCH_SIZE={BATCH_SIZE}  max_new_tokens={MAX_NEW_TOKENS_DEFAULT}")
print(f"  strict_cot             : BATCH_SIZE={BATCH_SIZE}  max_new_tokens={MAX_NEW_TOKENS_STRICT_COT}  ← larger budget for <think> block")
print(f"  self_consistency       : SC_BATCH_SIZE={SC_BATCH_SIZE}  (eff. {SC_BATCH_SIZE}\u00d7{SC_SAMPLES}={SC_BATCH_SIZE*SC_SAMPLES} seqs/pass)")
print(f"  MAX_LENGTH             : {MAX_LENGTH}")
print(f"  Memory: del+empty_cache inside every batch fn (\u00d72) + after every loop iteration")


## Cell 9 — Run Selected Strategy

Loads the model once, runs the single strategy in `STRATEGIES`, and stores results in `all_results`.

```
all_results = {
    strategy_name: {
        "predictions": [str, ...],    # per-row predicted letter
        "accuracy":    float,         # overall %
    }
}
```

In [ ]:
import time

# Results store
all_results = {}   # {strategy: {accuracy, predictions, raw_outputs}}
all_errors  = {}   # {strategy: [{idx, source, gold, predicted, strategy}, ...]}

# ── Free any model left in memory from a previous (possibly crashed) run ────────
# If a run crashed mid-way, the model was never freed and still occupies VRAM.
# Attempting to reload without clearing first causes device_map="auto" to spill
# to CPU, which BitsAndBytes 4-bit doesn't allow → ValueError.
try:
    free_model(model)
    print("[cleanup] previous 'model' freed")
except NameError:
    pass
try:
    del tokenizer
    print("[cleanup] previous 'tokenizer' freed")
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("[cleanup] CUDA cache cleared")
print()

# ── Load model (once) ──────────────────────────────────────────────────────────────────
print("=" * 65)
print(f"Loading model: {MODEL_ID}")
print("=" * 65)
model, tokenizer = load_model_4bit(MODEL_ID)

print()
print(f"Starting benchmark — {len(STRATEGIES)} strategies × {len(all_data)} questions")
if TEST_MODE:
    print(f"[TEST_MODE] {N_TEST} examples per dataset (before balancing)")
print()

# ── Run each strategy ─────────────────────────────────────────────────────────────────
for strategy in STRATEGIES:
    print("=" * 65)
    print(f"STRATEGY: {strategy}")
    print(f"  questions : {len(all_data)}")
    print("=" * 65)

    t0                       = time.time()
    preds, errors, raw_outputs = run_strategy(model, tokenizer, all_data, strategy)
    elapsed                  = time.time() - t0

    correct  = sum(1 for i, p in enumerate(preds) if p in all_data[i]["gold"])
    accuracy = correct / len(all_data) * 100

    all_results[strategy] = {"predictions": preds, "accuracy": accuracy, "raw_outputs": raw_outputs}
    all_errors[strategy]  = errors

    print(f"  Elapsed : {elapsed:.0f}s")
    print(f"  Overall accuracy [{strategy}]: {accuracy:.2f}%  ({correct}/{len(all_data)})")
    print(f"  Wrong predictions stored: {len(errors)}")
    print()

# ── Release model ──────────────────────────────────────────────────────────────────
free_model(model)
del tokenizer

print("=" * 65)
print("ALL STRATEGIES COMPLETE")
print("=" * 65)
for strat, res in all_results.items():
    n_err = len(all_errors.get(strat, []))
    print(f"  {strat:<22}  {res['accuracy']:.2f}%   ({n_err} errors logged)")


## Cell 10 — Results: Cartesian Comparison (Strategy × Dataset)

Full breakdown grid: every prompt strategy vs every dataset, plus overall accuracy per strategy.
With a single-strategy run, this will show one row — that is correct and expected.

In [ ]:
import pandas as pd

# ── Build per-source index maps ────────────────────────────────────────────────────────
sources     = sorted(set(r["source"] for r in all_data))
src_indices = {
    src: [i for i, r in enumerate(all_data) if r["source"] == src]
    for src in sources
}

# ── Cartesian table: rows = strategies, columns = datasets ─────────────────────
table_rows = []
for strategy, res in all_results.items():
    preds    = res["predictions"]
    row_dict = {"strategy": strategy}
    for src in sources:
        idxs          = src_indices[src]
        c             = sum(1 for i in idxs if preds[i] in all_data[i]["gold"])
        row_dict[src] = round(c / len(idxs) * 100, 1)
    c_all               = sum(1 for i, p in enumerate(preds) if p in all_data[i]["gold"])
    row_dict["OVERALL"] = round(c_all / len(all_data) * 100, 1)
    table_rows.append(row_dict)

df_cart = pd.DataFrame(table_rows).set_index("strategy")

# ── Print table ──────────────────────────────────────────────────────────────────────
print("=" * 80)
print("CARTESIAN RESULTS: Accuracy (%)  —  Strategy × Dataset")
print("=" * 80)
print(df_cart.to_string())
print()

print("-" * 80)
print("Best strategy per dataset:")
for col in df_cart.columns:
    best_strat = df_cart[col].idxmax()
    best_acc   = df_cart[col].max()
    print(f"  {col:<33}  {best_strat:<22}  {best_acc:.1f}%")

print()
print("-" * 80)
print("Best dataset per strategy:")
for strat in df_cart.index:
    src_cols = [c for c in df_cart.columns if c != "OVERALL"]
    best_ds  = df_cart.loc[strat, src_cols].idxmax()
    best_acc = df_cart.loc[strat, src_cols].max()
    print(f"  {strat:<22}  →  {best_ds:<33}  {best_acc:.1f}%")

print()
print("-" * 80)
print("Strategy ranking by OVERALL accuracy:")
ranked = df_cart["OVERALL"].sort_values(ascending=False)
for rank, (strat, acc) in enumerate(ranked.items(), 1):
    marker = "  ◄ best" if rank == 1 else ""
    print(f"  {rank}. {strat:<22}  {acc:.1f}%{marker}")

# ── Seaborn heatmap ─────────────────────────────────────────────────────────────────────
try:
    import seaborn as sns
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(max(8, len(sources) * 1.4), max(4, len(df_cart) * 0.7)))
    sns.heatmap(
        df_cart,
        annot=True, fmt=".1f",
        cmap="RdYlGn",
        linewidths=0.5,
        linecolor="white",
        vmin=max(0, df_cart.values.min() - 5),
        vmax=min(100, df_cart.values.max() + 5),
        ax=ax,
    )
    ax.set_title("Accuracy (%) — Strategy × Dataset", fontsize=13, pad=12)
    ax.set_xlabel("Dataset / OVERALL", fontsize=10)
    ax.set_ylabel("Prompt Strategy", fontsize=10)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig("finr1_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Heatmap saved: finr1_heatmap.png")
except ImportError:
    print("[heatmap] seaborn not installed — run: pip install seaborn matplotlib")

# ── Error analysis summary ─────────────────────────────────────────────────────────────
print()
print("=" * 80)
print("ERROR ANALYSIS — wrong predictions per strategy × source")
print("=" * 80)
error_header = f"  {'Strategy':<22}" + "".join(f"  {s[:12]:>12}" for s in sources) + f"  {'TOTAL':>6}"
print(error_header)
print("  " + "-" * (len(error_header) - 2))
for strategy in all_results:
    errs   = all_errors.get(strategy, [])
    by_src = {s: sum(1 for e in errs if e["source"] == s) for s in sources}
    row    = f"  {strategy:<22}"
    for s in sources:
        row += f"  {by_src[s]:>12}"
    row += f"  {len(errs):>6}"
    print(row)

## Cell 11 — Save Results

In [ ]:
import json
import pandas as pd

# ── Full predictions JSON ───────────────────────────────────────────────────────────────
output = {}
for strategy, res in all_results.items():
    preds       = res["predictions"]
    raw_outputs = res.get("raw_outputs", [None] * len(all_data))
    correct  = sum(1 for i, p in enumerate(preds) if p in all_data[i]["gold"])
    src_accs = {}
    for src in sources:
        idxs = src_indices[src]
        c    = sum(1 for i in idxs if preds[i] in all_data[i]["gold"])
        src_accs[src] = round(c / len(idxs) * 100, 2)
    output[strategy] = {
        # Summary stats
        "model":          MODEL_ID,
        "strategy":       strategy,
        "accuracy":       res["accuracy"],
        "correct":        int(correct),
        "total":          len(all_data),
        "balanced_size":  min_size,          # rows per dataset after balancing
        "test_mode":      TEST_MODE,
        "random_seed":    RANDOM_SEED,
        "timestamp":      __import__("datetime").datetime.utcnow().isoformat() + "Z",

        # Per-dataset accuracy breakdown
        "per_source": src_accs,             # {source_name: accuracy_pct}

        # Per-question detail — everything needed to compare runs offline
        # raw_output: full decoded model text (str for generation strategies,
        #             list[str] of SC_SAMPLES texts for self_consistency,
        #             null for logit-only strategies)
        "questions": [
            {
                "idx":        i,
                "source":     all_data[i]["source"],
                "question":   all_data[i]["question"],
                "choices":    all_data[i]["choices"],
                "gold":       all_data[i]["gold"],
                "predicted":  preds[i],
                "correct":    preds[i] in all_data[i]["gold"],
                "raw_output": raw_outputs[i],
            }
            for i in range(len(all_data))
        ],
    }

# Strategy-named file so parallel Kaggle runs don't overwrite each other
fname_json = f"finr1_{ACTIVE_STRATEGY}_results.json"
with open(fname_json, "w") as f:
    json.dump(output, f, indent=2)
print(f"Saved: {fname_json}")

# ── Cartesian accuracy CSV ───────────────────────────────────────────────────────────────
fname_csv = "finr1_prompt_engineering_cartesian.csv"
df_cart.reset_index().to_csv(fname_csv, index=False)
print(f"Saved: {fname_csv}")

# ── Error cases CSV ────────────────────────────────────────────────────────────────────
# Flat list of every mispredicted question across all strategies.
# Columns: strategy, idx, source, language, gold, predicted, question (truncated)
error_rows = []
for strategy, errs in all_errors.items():
    for e in errs:
        row_data = all_data[e["idx"]]
        error_rows.append({
            "strategy":  strategy,
            "idx":       e["idx"],
            "source":    e["source"],
            "language":  _SOURCE_LANG.get(e["source"], "?"),
            "gold":      "|".join(e["gold"]),
            "predicted": e["predicted"],
            "question":  row_data["question"][:200],
        })

df_errors = pd.DataFrame(error_rows)
fname_errors = "finr1_prompt_engineering_errors.csv"
df_errors.to_csv(fname_errors, index=False)
print(f"Saved: {fname_errors}  ({len(df_errors)} error rows across {len(all_results)} strategies)")

# ── Summary ───────────────────────────────────────────────────────────────────────────
print()
print("Answer format  : lowercase letter string ('a', 'b', 'c', ...)")
print("Correct        : predicted letter is in the gold list (multi-answer aware)")
print("Random baseline: ~25-33 % depending on choice count per question")
print()
print("raw_output field:")
print("  generation strategies  → str  (full model output incl. <think>/<answer> blocks)")
print("  self_consistency       → list[str]  (one string per SC sample)")
print("  logit strategies       → null  (no free-text generation)")
print()
print("Files written:")
print(f"  {fname_json:<50} full predictions + per-question detail (strategy-named)")
print(f"  {fname_csv:<50} cartesian accuracy table (strategy × dataset)")
print(f"  {fname_errors:<50} all mispredicted rows for error analysis")
